In [ ]:
# =========================================================
# 1. LOAD FITTED MODELS FROM NOTEBOOK 07
#
# Notebook 07 pickles the three PPML results plus the model
# dataframes to data/intermediate/ppml_models.pkl. Loading
# here keeps this notebook short and avoids re-running IRLS.
# Re-run 07 if the underlying data changed.
# =========================================================
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

warnings.filterwarnings('ignore')

MODEL_FILE = "../data/intermediate/ppml_models.pkl"

with open(MODEL_FILE, 'rb') as f:
    art = pickle.load(f)

res_pk        = art['res_pk']
res_prop      = art['res_prop']
res_prop_frac = art['res_prop_frac']
df_pk_model   = art['df_pk_model']
df_prop_model = art['df_prop_model']
demo_terms    = art['demo_terms']

print(f"Loaded models from {MODEL_FILE}")
print(f"  Placekey rows : {len(df_pk_model):,}")
print(f"  Property rows : {len(df_prop_model):,}")

In [ ]:
# =========================================================
# 3. DISTANCE-DECAY CURVES (gravity model intuition)
#
# The signature gravity-model behaviour is that visits fall
# off as distance grows. With log_dist as a predictor and a
# Poisson link, fitted visits ∝ distance^β (a power law).
# We plot the model's predicted visit count as distance
# varies, holding all other predictors at their sample mean,
# separately for FW and non-FW parks.
# =========================================================

def predict_grid(res, base_row, vary_col, vary_values):
    # Build a DataFrame whose rows differ only in `vary_col`, keeping
    # everything else at base_row's values, then call statsmodels.predict
    # for the linear-prediction expectation. Returns predicted visits.
    grid = pd.DataFrame([base_row] * len(vary_values))
    grid[vary_col] = vary_values
    return res.predict(grid)

dists = np.linspace(0.1, 20, 80)  # km; covers most of the empirical range

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, df, label in [
    (axes[0], res_pk,   df_pk_model,   'Placekey-level'),
    (axes[1], res_prop, df_prop_model, 'Property-level'),
]:
    # Build a "typical tract" / "typical park" base case from sample means.
    # log_area & log_pop stay at means; we vary log_dist on the x-axis.
    base = df[demo_terms + ['log_area', 'log_pop']].mean().to_dict()
    base['log_dist']  = 0.0  # placeholder, overwritten by predict_grid

    for is_nat, color, name in [(0, 'steelblue', 'Standard parks'),
                                (1, 'darkgreen', 'Forever Wild')]:
        b = dict(base)
        b['is_nature'] = is_nat
        # Predict at every distance value in the grid
        log_d = np.log(np.clip(dists, 1e-3, None))
        preds = predict_grid(res, b, 'log_dist', log_d)
        ax.plot(dists, preds, color=color, label=name, linewidth=2)

    ax.set_yscale('log')
    ax.set_xlabel('Distance (km)')
    ax.set_ylabel('Predicted visits (log scale)')
    ax.set_title(f'Distance decay — {label}')
    ax.legend()
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 4. DEMOGRAPHIC GRADIENTS — predicted visits vs each demo
#
# For each demographic predictor, vary it over its empirical
# range (5th–95th percentile) and plot predicted visits at
# both is_nature=0 and is_nature=1, holding everything else
# at the sample mean. The vertical gap between the two
# lines for a given x is the marginal effect of the FW
# designation at that demographic level — the "interaction".
# =========================================================

# Plot grid: 3x3 panel for the 9 demographic predictors
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
axes = axes.flatten()

# Use the property-level model for this — coefficients are more stable
# (less noise from idiosyncratic POIs) and counts are more comparable
# across tracts.
res = res_prop
df  = df_prop_model

base = df[demo_terms + ['log_dist', 'log_area', 'log_pop']].mean().to_dict()

for ax, demo in zip(axes, demo_terms):
    # 5th–95th percentile range avoids extreme tails distorting the plot
    lo, hi = df[demo].quantile([0.05, 0.95])
    grid_vals = np.linspace(lo, hi, 60)

    for is_nat, color, name in [(0, 'steelblue', 'Standard'),
                                (1, 'darkgreen', 'Forever Wild')]:
        b = dict(base)
        b['is_nature'] = is_nat
        preds = predict_grid(res, b, demo, grid_vals)
        ax.plot(grid_vals, preds, color=color, label=name, linewidth=2)

    ax.set_xlabel(demo)
    ax.set_ylabel('Predicted visits')
    ax.set_title(demo)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Demographic gradients — Property-level PPML\n(other predictors held at sample mean)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 5. NATURE-FRAC GRADIENT — model #3 only
#
# Model #3 lets the demographic effect change continuously
# with how much of a park is Forever Wild. For each demo,
# we draw a family of curves at nature_frac ∈ {0, .25, .5,
# .75, 1.0}: each line shows the demographic gradient at
# that level of "naturalness". Spread between lines = the
# strength of the interaction in model #3.
# =========================================================

frac_levels = np.linspace(0, 1, 5)
cmap = plt.cm.YlGn  # yellow→green: more green = more natural

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
axes = axes.flatten()

base = df_prop_model[demo_terms + ['log_dist', 'log_area', 'log_pop']].mean().to_dict()

for ax, demo in zip(axes, demo_terms):
    lo, hi = df_prop_model[demo].quantile([0.05, 0.95])
    grid_vals = np.linspace(lo, hi, 60)

    for nf in frac_levels:
        b = dict(base)
        b['nature_frac'] = nf
        preds = predict_grid(res_prop_frac, b, demo, grid_vals)
        # Skip nf=0 in the colormap edge so the lightest line is still visible
        ax.plot(grid_vals, preds, color=cmap(0.2 + 0.8 * nf), linewidth=2,
                label=f'nature_frac={nf:.2f}')

    ax.set_xlabel(demo)
    ax.set_ylabel('Predicted visits')
    ax.set_title(demo)
    ax.grid(alpha=0.3)
    if demo == demo_terms[0]:
        ax.legend(fontsize=7, loc='best')

fig.suptitle('Demographic gradients across the nature_frac continuum (model #3)\n'
             'lighter = manicured park, darker = entirely natural',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 6. COEFFICIENT HEATMAP — all three models side by side
#
# Sign and magnitude tell different stories. The heatmap
# uses sign+magnitude in cell colour, with significance
# stars overlaid. Models #1, #2, #3 sit in adjacent columns
# so any predictor that flips sign across them is obvious.
# =========================================================

def coef_table(res, label, nature_var):
    # Pull params + p-values into long form. We standardise the variable
    # name "is_nature" -> generic "nature" so the row labels align across
    # binary and continuous nature models (model #3 uses nature_frac).
    rows = []
    for name in res.params.index:
        clean = name.replace(nature_var, 'nature')
        rows.append({
            'variable': clean,
            'coef':     res.params[name],
            'pvalue':   res.pvalues[name],
            'model':    label,
        })
    return pd.DataFrame(rows)

ct = pd.concat([
    coef_table(res_pk,        '#1 Placekey',     'is_nature'),
    coef_table(res_prop,      '#2 Property',     'is_nature'),
    coef_table(res_prop_frac, '#3 Property frac','nature_frac'),
])

# Pivot to a model x variable matrix. drop the intercept since it's
# scale-only and dwarfs the others on a heatmap.
mat = ct.pivot(index='variable', columns='model', values='coef').drop('Intercept', errors='ignore')

# Order rows so main effects come first, then interactions
main = ['nature', 'log_dist', 'log_area', 'log_pop'] + demo_terms
inter = [f'nature:{d}' for d in demo_terms]
ordered = [v for v in (main + inter) if v in mat.index]
mat = mat.reindex(ordered)

# Symmetric color scale around 0 — easier to read sign at a glance
vmax = mat.abs().max().max()

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(mat, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            cbar_kws={'label': 'PPML coefficient (log-link)'},
            ax=ax, linewidths=0.4, linecolor='white')

# Overlay significance stars on top of each annotation
pmat = ct.pivot(index='variable', columns='model', values='pvalue').reindex(ordered)
for i, var in enumerate(ordered):
    for j, model in enumerate(mat.columns):
        p = pmat.loc[var, model]
        if pd.notna(p):
            star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            if star:
                ax.text(j + 0.5, i + 0.85, star, ha='center', va='center',
                        fontsize=8, color='black')

ax.set_title('PPML coefficients across the three models\n(* p<0.05, ** p<0.01, *** p<0.001)', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 7. PREDICTED vs OBSERVED — model fit diagnostic
#
# A well-fitting Poisson model puts most points along the
# 45-degree line. We bin observations into deciles of the
# predicted mean and compare median observed counts in each
# bin — this smooths out the heteroskedasticity that shows
# up as a noisy point cloud at the per-row level.
# =========================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, res, df, name in [
    (axes[0], res_pk,        df_pk_model,   '#1 Placekey'),
    (axes[1], res_prop,      df_prop_model, '#2 Property (binary)'),
    (axes[2], res_prop_frac, df_prop_model, '#3 Property (frac)'),
]:
    pred = res.predict(df)
    obs  = df['visits'].values

    # Decile-binned medians give a robust view of calibration vs the
    # raw scatter (which is dominated by Poisson variance at low counts)
    dec = pd.qcut(pred, q=20, labels=False, duplicates='drop')
    bin_df = pd.DataFrame({'pred': pred, 'obs': obs, 'bin': dec})
    summary = bin_df.groupby('bin').agg(pred=('pred', 'mean'),
                                        obs_med=('obs', 'median'),
                                        obs_q25=('obs', lambda x: np.quantile(x, 0.25)),
                                        obs_q75=('obs', lambda x: np.quantile(x, 0.75)))

    ax.fill_between(summary['pred'], summary['obs_q25'], summary['obs_q75'],
                    color='steelblue', alpha=0.25, label='25th–75th pct')
    ax.plot(summary['pred'], summary['obs_med'], 'o-', color='steelblue', label='Median observed')

    # Reference: perfect calibration
    lo = max(min(summary['pred'].min(), 1), 1)
    hi = max(summary['pred'].max(), summary['obs_med'].max())
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.6, label='y = x')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Predicted visits (decile-binned mean)')
    ax.set_ylabel('Observed visits')
    ax.set_title(name)
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 8. ELASTICITIES (log-link interpretation)
#
# In a log-link Poisson, the coefficient on a continuous
# predictor is approximately the elasticity of E[visits]
# w.r.t. that predictor when the predictor is itself in
# logs (log_dist, log_area, log_pop). For other predictors
# (income, demos), exp(β·Δx) gives the multiplicative
# change in expected visits per unit increase.
#
# We translate model coefficients into "% change in visits
# for a 10-percentage-point shift" for each demographic and
# both nature regimes.
# =========================================================

# Per-10pp demographic shift, separately for FW and non-FW parks (model #2)
delta = 0.10  # 10 percentage points
rows = []

for d in demo_terms:
    if d == 'median_income':
        # $10k step makes more sense than 10% for median income
        step = 10000
        unit = '$10k income'
    else:
        step = delta
        unit = '+10pp'

    # Marginal coefficient at is_nature=0 is just the main effect.
    # At is_nature=1 it's main + interaction.
    main  = res_prop.params[d]
    inter = res_prop.params.get(f'is_nature:{d}', 0.0)

    pct_non = (np.exp(main * step) - 1) * 100
    pct_fw  = (np.exp((main + inter) * step) - 1) * 100

    rows.append({'demographic': d, 'unit': unit,
                 'std_park_pct': pct_non, 'fw_park_pct': pct_fw})

elas = pd.DataFrame(rows)

# Side-by-side bar chart of the same shift in standard vs FW parks
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(elas))
w = 0.4
ax.bar(x - w/2, elas['std_park_pct'], w, label='Standard parks', color='steelblue')
ax.bar(x + w/2, elas['fw_park_pct'],  w, label='Forever Wild',  color='darkgreen')
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels([f"{r.demographic}\n({r.unit})" for r in elas.itertuples()],
                   rotation=30, ha='right', fontsize=9)
ax.set_ylabel('% change in expected visits')
ax.set_title('Marginal effect on expected visits — standard vs Forever Wild parks\n'
             '(Property-level model #2; other predictors held at sample mean)',
             fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(elas.to_string(index=False))

In [ ]:
# =========================================================
# 9. RESIDUAL ANALYSIS — where does the model miss?
#
# Pearson residuals = (obs - pred) / sqrt(pred). Under a
# correctly specified Poisson model these have unit
# variance. Large positive residuals are tract-property
# pairs the model under-predicts; large negative ones are
# over-predictions. We rank top "surprises" of each kind
# for inspection.
# =========================================================

# Use property-level model #2 because pairs are easier to interpret
df_resid = df_prop_model.copy()
df_resid['pred'] = res_prop.predict(df_resid)
df_resid['pearson'] = (df_resid['visits'] - df_resid['pred']) / np.sqrt(df_resid['pred'])

# Histogram + qq-style diagnostic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of residuals — should be roughly bell-shaped if model fits
axes[0].hist(df_resid['pearson'].clip(-15, 15), bins=80, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Pearson residual (clipped to ±15)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual distribution — Property model #2')
axes[0].grid(alpha=0.3)

# Residual vs predicted: look for systematic patterns. A wedge shape
# would indicate over- or under-dispersion at certain count levels.
axes[1].scatter(df_resid['pred'], df_resid['pearson'],
                s=4, alpha=0.2, color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Predicted visits (log)')
axes[1].set_ylabel('Pearson residual')
axes[1].set_title('Residual vs predicted')
axes[1].set_ylim(-30, 30)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Top 10 under-predictions (model says low, observed high) — these are
# tracts that visit a property far more than the model expects, e.g.
# specific marquee parks for nearby neighborhoods.
print("\nTOP 10 UNDER-PREDICTIONS (visits > pred):")
print(df_resid.nlargest(10, 'pearson')[['tract_i', 'gis_prop_num', 'property_name',
                                         'visits', 'pred', 'pearson']]
        .round(2).to_string(index=False))

# Top 10 over-predictions: model says many visits, observed is low —
# usually small or low-traffic parks where the gravity model overshoots.
print("\nTOP 10 OVER-PREDICTIONS (pred > visits):")
print(df_resid.nsmallest(10, 'pearson')[['tract_i', 'gis_prop_num', 'property_name',
                                          'visits', 'pred', 'pearson']]
        .round(2).to_string(index=False))

In [ ]:
# =========================================================
# 10. NATURE-FRAC RESPONSE CURVE
#
# What is the model #3 prediction for visits as a property
# becomes "more natural" (nature_frac sweep), at fixed
# tract demographics? Three reference tracts are chosen
# spanning the income distribution.
# =========================================================

# Anchor the sweep at three tract types: low / median / high income
qs = df_prop_model['median_income'].quantile([0.1, 0.5, 0.9])
labels = ['Low-income tract (10th pct)',
          'Median-income tract',
          'High-income tract (90th pct)']
colors = ['#d6604d', '#666666', '#4393c3']

base = df_prop_model[demo_terms + ['log_dist', 'log_area', 'log_pop']].mean().to_dict()
nf_grid = np.linspace(0, 1, 50)

fig, ax = plt.subplots(figsize=(10, 6))
for q, label, color in zip(qs, labels, colors):
    b = dict(base)
    b['median_income'] = q
    preds = predict_grid(res_prop_frac, b, 'nature_frac', nf_grid)
    ax.plot(nf_grid, preds, color=color, linewidth=2.5, label=f'{label} (${q:,.0f})')

ax.set_xlabel('nature_frac (share of property that is Forever Wild)')
ax.set_ylabel('Predicted visits')
ax.set_title('Predicted visits as a function of "naturalness", by tract income\n'
             '(Property model #3; other predictors at sample mean)', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 11. ATTENDANCE RATE — visits per 1k tract residents
#
# Raw visit counts confound population size with park
# preference. Per-capita rates are easier to interpret as
# a "propensity to visit". We compute mean visits-per-1k
# residents by FW status × demographic decile so we can see
# how visit rates vary across the demographic distribution.
# =========================================================

# Use the property-level data and rescale visits per 1k people
df_rate = df_prop_model.copy()
df_rate['visits_per_1k'] = df_rate['visits'] / (df_rate['total_population'] / 1000)

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
axes = axes.flatten()

for ax, demo in zip(axes, demo_terms):
    # Decile-bin tracts on each demographic, compute mean rate per bin
    bins = pd.qcut(df_rate[demo], q=10, labels=False, duplicates='drop')
    summary = (df_rate.assign(bin=bins)
                      .groupby(['bin', 'is_nature'])['visits_per_1k']
                      .mean()
                      .reset_index())

    for is_nat, color, name in [(0, 'steelblue', 'Standard'),
                                (1, 'darkgreen', 'Forever Wild')]:
        sub = summary[summary['is_nature'] == is_nat]
        ax.plot(sub['bin'] + 1, sub['visits_per_1k'], 'o-', color=color, label=name, linewidth=2)

    ax.set_xlabel(f'{demo} decile')
    ax.set_ylabel('Visits per 1k residents')
    ax.set_title(demo)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Per-capita visit rates by tract demographic decile and FW status\n'
             '(empirical, not modeled — based on observed flows)',
             fontsize=12)
plt.tight_layout()
plt.show()